# RAG Demonstration

In [4]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import random
import re
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from typing import List
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json

In [5]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [3]:
df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/synq.csv")
queries = df['QUERY'].tolist()
passages = df['PASSAGE'].tolist()
df.head(20)

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,NOTE,QUERY,PASSAGE
0,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Does the patient have a history of tuberculosi...,Admission Date: [**2151-7-16**] Dischar...
1,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,HEAD CT: Head CT showed no intracranial hemor...,Has the patient had any previous imaging studi...,HEAD CT: Head CT showed no intracranial hemor...
2,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,[**MD Number(1) 1776**]\n\nDictated By:[**Hosp...,What is the patient's date of birth?,[**MD Number(1) 1776**]\n\nDictated By:[**Hosp...
3,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,Admission Date: [**2118-6-2**] Discharg...,Does the patient have a history of oxygen ther...,Admission Date: [**2118-6-2**] Discharg...
4,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,"Two days prior to admission,\nshe was started ...",What was the patient's oxygen saturation level...,"Two days prior to admission,\nshe was started ..."
5,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,She was not able to be weaned off of this\ndes...,What is the patient's current respiratory stat...,She was not able to be weaned off of this\ndes...
6,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,Is positive for the following:\nChest pressure...,Does the patient have a history of heart disease?,Is positive for the following:\nChest pressure...
7,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,Last pulmonary function tests in [**2117-11-3*...,Does the patient have a history of chronic obs...,Last pulmonary function tests in [**2117-11-3*...
8,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,"The FVC, however, does significantly improve w...",Has the patient ever been diagnosed with or tr...,"The FVC, however, does significantly improve w..."
9,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,MRI of the head in [**2114-11-4**]\ndemonstrat...,Has the patient experienced any previous episo...,MRI of the head in [**2114-11-4**]\ndemonstrat...


### Example of a Query:

In [6]:
print(queries[0])

Does the patient have a history of tuberculosis prior to this admission?


### Example Passage:

In [7]:
print(passages[0])

Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]


Service:
ADDENDUM:

RADIOLOGIC STUDIES:  Radiologic studies also included a chest
CT, which confirmed cavitary lesions in the left lung apex
consistent with infectious process/tuberculosis. This also
moderate-sized left pleural effusion.


### For Patient:

In [9]:
print('Subject ID: ' + str(df['SUBJECT_ID'].iloc[0]))

Subject ID: 22532


### Loading the vector DB, index and json

In [10]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage_2.index")

# Load Metadata
metadata = []
with open("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage_metadata_2.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

### Loading the Trained Query Encoder

In [11]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained("bert-base-uncased")

query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/MEDRAG/model_weights/query_encoder_2"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512
        ) #.to("cuda")

    outputs = query_encoder(**inputs)
    cls_embeddings = outputs.last_hidden_state[:, 0] 

    embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, 0)

### Testing custom question

In [12]:
question = 'Subject ID: ' + str(df['SUBJECT_ID'].iloc[0]) + '\n'  +  queries[0]
print(question)

Subject ID: 22532
Does the patient have a history of tuberculosis prior to this admission?


In [13]:
# Embedding Query
query_emb = encode_query([question]).detach().cpu().numpy()

In [14]:
# Matching for the top K=5 highest scores
K = 10
scores, ids = index.search(query_emb, K)

In [15]:
print(scores)

[[295.53992 295.1206  294.99463 294.29846 292.26538 292.0077  291.98004
  291.96265 291.86887 291.80972]]


In [16]:
candidates = [metadata[i]["text"] for i in ids[0]]
for i, passage in enumerate(candidates):
    print(passage)
    print("\n\n\n") 

Subject ID: 27650
lisinopril 5 mg po qday


Discharge Disposition:
Extended Care

Facility:
[**Location (un) 4047**] Nursing & Rehabilitation Center - [**Location (un) 4047**]

Discharge Diagnosis:
1. gallstone pancreatitis s/p ERCP with stent placement
2. E.coli sepsis, resolved
3.




Subject ID: 66079
# Pus around HD line.-no pus was seen around HD line on the
floors. With no fluctuance, draining fluid, or erythema. Vancomycin 1.5g with hD per renal fellow- discontinued per
renal. F/u cultures- no growth to date. Bacitracin admin. only
with dialysis to HD entry site recommended. . # ESRD. - continued home meds.Ordered Folic acid home dose. Contnued
Sevelemer 800mg TID per renal. . # HTN.




Subject ID: 361
Because of the risks involved in removal of the IVC including
possible showering of emboli and difficulty removing due to
fibrosis, the decision was made to leave the IVC filter in
place, and as a result it will become a permanent IVC filter. . The patient was started on Coumadin

# Other infromation

### At this time:

In [33]:
# Synthetic Dataset length:
print(len(df))

28431


In [34]:
# Additional Synthetic Dataset Length:
add_df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/add_synq.csv")
print(len(add_df))

79825


In [35]:
# add_df.head(20)